# Tutorial: Analysis of CASSCF Solutions

---

This tutorial shows how to interpret the results of a CASSCF calculation using different types of orbitals.
This tutorial assumes that you are already familiar with CASSCF claculations in Forte2.

### 💻 Importing the relevant modules from Forte2

To begin we will import a few components from Forte2:

In [1]:
from pathlib import Path
import numpy as np

HAVE_MPL = True
try:
    import matplotlib.pyplot as plt
except ImportError as e:
    print(f"You likely need to install matplotlib: see original error message: {e}")
    HAVE_MPL = False

from forte2 import MCOptimizer, RHF, CISolver, State, System, write_orbital_cubes, AVAS
from forte2.base_classes.params import DavidsonLiuParams

forte2: using 10 threads for parallel sections
[mods_manager] loading mod determinant_printing from /Users/fevange/.forte2/mods
[mods_manager] failed to load mod determinant_printing from /Users/fevange/.forte2/mods: partially initialized module 'forte2' from '/Users/fevange/Source/forte2-dev-2/forte2/__init__.py' has no attribute 'Determinant' (most likely due to a circular import)


In [4]:
xyz = f"""
C 1.400000 -0.000000 -0.000000
C 0.700000 1.212000 -0.000000
C -0.700000 1.212000 -0.000000
C -1.400000 -0.000000 -0.000000
C -0.700000 -1.212000 -0.000000
C 0.700000 -1.212000 -0.000000
H 2.480000 -0.000000 -0.000000
H 1.240000 2.150000 -0.000000
H -1.240000 2.150000 -0.000000
H -2.480000 -0.000000 -0.000000
H -1.240000 -2.150000 -0.000000
H 1.240000 -2.150000 -0.000000
"""

system = System(
    xyz=xyz,
    basis_set="cc-pVDZ",
    auxiliary_basis_set="cc-pVTZ-JKFIT",
    symmetry=False,
)
rhf = RHF(charge=0, e_tol=1.0e-12)(system)

avas = AVAS(subspace=["C(2pz)"],selection_method="separate",num_active_docc=3,num_active_uocc=3)(rhf)

singlet = State(system=system, multiplicity=1, ms=0.0)

cas_solver = CISolver(
    states=singlet
)

casscf_default = MCOptimizer(cas_solver, final_orbitals="ibo_atomic")(avas)

casscf_default.run()

Point group symmetry detection not performed. Running in C1 symmetry.
Principal Atomic Positions (a.u.):
   C   2.64561657   -0.00000000   -0.00000000
   C   1.32280829   2.29034806   -0.00000000
   C   -1.32280829   2.29034806   -0.00000000
   C   -2.64561657   -0.00000000   -0.00000000
   C   -1.32280829   -2.29034806   -0.00000000
   C   1.32280829   -2.29034806   -0.00000000
   H   4.68652079   -0.00000000   -0.00000000
   H   2.34326039   4.06291117   -0.00000000
   H   -2.34326039   4.06291117   -0.00000000
   H   -4.68652079   -0.00000000   -0.00000000
   H   -2.34326039   -4.06291117   -0.00000000
   H   2.34326039   -4.06291117   -0.00000000
Parsed 12 atoms with basis set of 114 functions.
  Max eigenvalue: 5.885e+00
  Min eigenvalue: 3.734e-04
  Condition number: 1.576e+04
  Inverse condition number: 6.344e-05
  Number of discarded eigenvalues: 0
  Number of kept eigenvalues: 114
  Largest discarded eigenvalue: 0.000e+00
  Smallest kept eigenvalue: 3.734e-04

AVAS: parsing th

MCOptimizer(requires={'mos', 'system'}, requires_attrs={'two_component': False}, provides={'mo_space', 'mos', 'system'}, two_component=False, executed=True, called=True, ci_solver=CISolver(requires={'mos', 'system'}, requires_attrs={'two_component': False}, provides={'mo_space', 'mos', 'system'}, two_component=False, executed=True, called=True, states=State(multiplicity=1, ms=0.0, nel=42, system=System(atoms=[[6, array([ 2.64561657, -0.        , -0.        ])], [6, array([ 1.32280829,  2.29034806, -0.        ])], [6, array([-1.32280829,  2.29034806, -0.        ])], [6, array([-2.64561657, -0.        , -0.        ])], [6, array([-1.32280829, -2.29034806, -0.        ])], [6, array([ 1.32280829, -2.29034806, -0.        ])], [1, array([ 4.68652079, -0.        , -0.        ])], [1, array([ 2.34326039,  4.06291117, -0.        ])], [1, array([-2.34326039,  4.06291117, -0.        ])], [1, array([-4.68652079, -0.        , -0.        ])], [1, array([-2.34326039, -4.06291117, -0.        ])], [1, 

When expressed in the semicanonical orbital basis, the CASSCF wavefunction contains many leading determinants with significant contributions to the wavefunction.
This can be seen in the following table printed out in the output, which shows the top determinants in the CASSCF wavefunction along with their CI coefficients.
```md
Top determinants:
===========================================================================
Contrib.  #1           #2           #3           #4           #5           
---------------------------------------------------------------------------
Root 0    |222b20a0>   |222a20b0>   |2222ba00>   |2222ab00>   |222aabb0>   
          -0.287714    -0.287714    +0.287705    +0.287705    +0.233169    
===========================================================================
```

## Cube files for all orbitals

The following code block generates cube files for all orbitals in the calculation.

In [3]:
write_orbital_cubes(
    system=system,
    C=casscf_default.mos.C[0],
    indices=[18,19,20,21,22,23],
    filepath=Path(f"benzene_cubes/ibo"),
)


Generating cube files with the following parameters:
  Grid origin: (-8.687, -8.063, -4.000)
  Grid points: 87 x 81 x 40 points.
  Scaled axes: [(0.2, 0, 0), (0, 0.2, 0), (0, 0, 0.2)]
  Orbitals: [18, 19, 20, 21, 22, 23]

